## Feature Engineering

#### Import packages

In [6]:
import pandas as pd
import numpy as np

#### Load cleaned data

In [7]:
data = pd.read_parquet(
    "../data/processed/train_transaction_cleaned.parquet", 
    engine="fastparquet"
)

#### Create a copy

In [13]:
features_data = data.copy()

#### Create time-based features
`TransactionsDT` is a time offset in seconds. We can use it to extract useful relative time features.

In [15]:
features_data["transaction_hour"] = (
    features_data["TransactionDT"] / 3600
) % 24

features_data["transaction_day"] = (
    features_data["TransactionID"] / (3600 * 24)
).astype(int)

features_data["transaction_week"] = (
    features_data["transaction_day"] / 7
).astype(int)

# Add a night transaction flag
features_data["is_night_transaction"] = (
    (features_data["transaction_hour"] < 6) |
    (features_data["transaction_hour"] > 22)
).astype(int)

#### Create amount-based features

In [17]:
features_data["transaction_amount_log"] = np.log1p(
    features_data["TransactionAmt"]
)

# Create rounded amount indicators
features_data["is_round_amount"]  = (
    features_data["transaction_amount_log"] % 1 == 0
).astype(int)

features_data["is_high_amount"] = (
    features_data["TransactionAmt"] > 
    features_data["TransactionAmt"].quantile(0.95)
).astype(int)